# 01 — Évaluation MEDCON sur le dictionnaire nettoyé

**Pipeline :**
1. Clone du repo
2. Upload des fichiers data
3. MEDCON groupé sur GPT-4o (WMT24)
4. BLEU + COMET
5. Analyse des erreurs de terminologie
6. Visualisations + export

## 0A. Clone du repo

In [1]:
!git clone https://github.com/11Maroua/Evaluating_medical_machine_translation.git

Cloning into 'Evaluating_medical_machine_translation'...
remote: Enumerating objects: 55, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 55 (delta 15), reused 51 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (55/55), 13.14 MiB | 3.95 MiB/s, done.
Resolving deltas: 100% (15/15), done.
Updating files: 100% (13/13), done.


## 0B. Fix environnement + installation



In [ ]:
import subprocess, os

# Fix conflit numpy (binary incompatibility)
subprocess.run(['pip', 'install', '-q', '--upgrade', 'numpy'], check=True)
# Install des dépendances
subprocess.run(['pip', 'install', '-q', 'pyahocorasick', 'sacrebleu', 'unbabel-comet'], check=True)

print('Installation terminée — redémarrage du runtime...')
os.kill(os.getpid(), 9)  # restart propre

## 0C. Upload des fichiers data


In [2]:
import os
from google.colab import files

# Ancrage dans le repo — toutes les cellules suivantes en héritent
REPO = '/content/Evaluating_medical_machine_translation'
os.chdir(REPO)
print(f'Répertoire de travail : {os.getcwd()}')

os.makedirs('data', exist_ok=True)
os.makedirs('results', exist_ok=True)

uploaded = files.upload()  # cleaned_mesh_snomed_dico.json + corpus_WMT24.json
for fname in uploaded:
    dest = f'data/{fname}'
    os.replace(fname, dest)
    print(f'  → {dest}')

Répertoire de travail : /content/Evaluating_medical_machine_translation


Saving cleaned_mesh_snomed_dico.json to cleaned_mesh_snomed_dico.json
Saving corpus_WMT24.json to corpus_WMT24.json
  → data/cleaned_mesh_snomed_dico.json
  → data/corpus_WMT24.json


## 1. Imports

In [3]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from collections import Counter
from scipy.stats import pearsonr
import sacrebleu

os.chdir('/content/Evaluating_medical_machine_translation')
sys.path.insert(0, 'src')
from medcon import build_pairs, build_automaton, medcon_grouped

print('Imports OK')
print(f'numpy  : {np.__version__}')
print(f'pandas : {pd.__version__}')

Imports OK
numpy  : 1.26.4
pandas : 2.2.2


## 2. Chargement des données

In [4]:
with open('data/cleaned_mesh_snomed_dico.json', encoding='utf-8') as f:
    raw_dict = json.load(f)

with open('data/corpus_WMT24.json', encoding='utf-8') as f:
    test_docs = json.load(f)

print(f'Dictionnaire nettoyé : {len(raw_dict):,} entrées EN')
print(f'Corpus WMT24         : {len(test_docs)} documents')
print(f'Clés corpus          : {list(test_docs[0].keys())}')

print('\n--- Aperçu dictionnaire (5 premières entrées) ---')
for en_term, fr_terms in list(raw_dict.items())[:5]:
    print(f'  {en_term!r:40s} -> {fr_terms}')

Dictionnaire nettoyé : 2,515 entrées EN
Corpus WMT24         : 49 documents
Clés corpus          : ['doc_id', 'text_en', 'translation_fr', 'gpt_translation']

--- Aperçu dictionnaire (5 premières entrées) ---
  'diagnostic imaging'                     -> ['Imagerie diagnostique', 'imagerie', 'imagerie diagnostique', 'imagerie médicale']
  'abnormalities'                          -> ['malformations']
  'administration & dosage'                -> ['administration et posologie']
  'adverse effects'                        -> ['effets indésirables']
  'analogs & derivatives'                  -> ['analogues et dérivés']


## 3. Construction des automates Aho-Corasick

In [9]:
pairs        = build_pairs(raw_dict)
automaton_en = build_automaton(pairs, 'en')
automaton_fr = build_automaton(pairs, 'fr')

print(f'Paires : {len(pairs):,}  |  Termes EN : {len(automaton_en):,}  |  Termes FR : {len(automaton_fr):,}')

# Test rapide
r_test = medcon_grouped(
    'The patient had a myocardial infarction and was treated for hypertension.',
    'Le patient a subi un infarctus du myocarde et a été traité pour hypertension artérielle.',
    pairs, automaton_en, automaton_fr
)


Paires : 2,515  |  Termes EN : 2,515  |  Termes FR : 3,186


## 4. MEDCON groupé sur GPT-4o

In [10]:
print('=' * 80)
print('MEDCON GROUPÉ -- GPT-4o  (dictionnaire nettoyé)')
print('=' * 80)

results_gpt = []
for i, doc in enumerate(tqdm(test_docs, desc='MEDCON')):
    r = medcon_grouped(
        doc['text_en'],
        doc['gpt_translation'],
        pairs, automaton_en, automaton_fr
    )
    results_gpt.append({
        'doc_id'           : i,
        'source_en'        : doc['text_en'],
        'gpt_fr'           : doc['gpt_translation'],
        'ref_fr'           : doc.get('translation_fr', ''),
        'medcon_f1'        : r['f1'],
        'medcon_precision' : r['precision'],
        'medcon_recall'    : r['recall'],
        'n_expected'       : r['n_expected'],
        'n_found'          : r['n_found'],
        'n_match'          : r['n_match'],
        'en_concepts'      : r['en_concepts'],
        'matched'          : r['matched'],
        'missed'           : r['missed'],
        'extra'            : r['extra'],
    })

df_gpt = pd.DataFrame(results_gpt)

print(f'\n  Precision : {df_gpt["medcon_precision"].mean():.3f}')
print(f'  Recall    : {df_gpt["medcon_recall"].mean():.3f}')
print(f'  F1        : {df_gpt["medcon_f1"].mean():.3f}')
print(f'\n  Paires attendues (moy.) : {df_gpt["n_expected"].mean():.1f}')
print(f'  Paires trouvées  (moy.) : {df_gpt["n_found"].mean():.1f}')
print(f'  Paires matchées  (moy.) : {df_gpt["n_match"].mean():.1f}')

MEDCON GROUPÉ -- GPT-4o  (dictionnaire nettoyé)


MEDCON:   0%|          | 0/49 [00:00<?, ?it/s]


  Precision : 0.418
  Recall    : 0.388
  F1        : 0.398

  Paires attendues (moy.) : 0.8
  Paires trouvées  (moy.) : 0.8
  Paires matchées  (moy.) : 0.6


In [14]:
# Voir quels termes sont effectivement matchés sur quelques docs
for i, row in df_gpt.head(10).iterrows():
    print(f"Doc {i} | attendues={row['n_expected']} | concepts: {row['en_concepts']}")

Doc 0 | attendues=0 | concepts: []
Doc 1 | attendues=0 | concepts: []
Doc 2 | attendues=0 | concepts: []
Doc 3 | attendues=0 | concepts: []
Doc 4 | attendues=0 | concepts: []
Doc 5 | attendues=0 | concepts: []
Doc 6 | attendues=2 | concepts: ['acinetobacter', 'rickettsiales']
Doc 7 | attendues=1 | concepts: ['genetics']
Doc 8 | attendues=0 | concepts: []
Doc 9 | attendues=4 | concepts: ['cardiorespiratory fitness', 'covid-19', 'cardiometabolic risk factors', 'mortality']


## 5. BLEU

In [12]:
print('Calcul BLEU...')
for i, doc in enumerate(tqdm(test_docs, desc='BLEU')):
    bleu = sacrebleu.sentence_bleu(doc['gpt_translation'], [doc['translation_fr']]).score
    df_gpt.loc[i, 'bleu'] = bleu

print(f'BLEU moyen : {df_gpt["bleu"].mean():.2f}')

Calcul BLEU...


BLEU:   0%|          | 0/49 [00:00<?, ?it/s]

BLEU moyen : 49.15


## 6. COMET (optionnel — nécessite GPU)

In [13]:
USE_COMET = False
try:
    import torch
    from comet import download_model, load_from_checkpoint

    print('Chargement COMET (wmt22-comet-da)...')
    comet_model = load_from_checkpoint(download_model('Unbabel/wmt22-comet-da'))

    comet_data = [
        {'src': doc['text_en'], 'mt': doc['gpt_translation'], 'ref': doc['translation_fr']}
        for doc in test_docs
    ]
    comet_output = comet_model.predict(
        comet_data,
        batch_size=8,
        gpus=1 if torch.cuda.is_available() else 0
    )
    for i, score in enumerate(comet_output.scores):
        df_gpt.loc[i, 'comet'] = float(score)

    USE_COMET = True
    print(f'COMET moyen : {df_gpt["comet"].mean():.3f}')

except Exception as e:
    print(f'COMET indisponible : {e}')
    df_gpt['comet'] = np.nan

Chargement COMET (wmt22-comet-da)...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

checkpoints/model.ckpt:   0%|          | 0.00/2.32G [00:00<?, ?B/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecat

COMET moyen : 0.875


## 7. Comparaison MEDCON / BLEU / COMET

In [15]:
print('=' * 80)
print('COMPARAISON MEDCON vs BLEU vs COMET  (GPT-4o)')
print('=' * 80)

print(f"\n{'Métrique':<14} {'Moyenne':>9} {'Std':>9} {'Min':>9} {'Max':>9}")
print('-' * 52)
for name, col in [('MEDCON F1', 'medcon_f1'), ('BLEU', 'bleu'), ('COMET', 'comet')]:
    if col in df_gpt and df_gpt[col].notna().any():
        s = df_gpt[col].dropna()
        print(f'{name:<14} {s.mean():>9.3f} {s.std():>9.3f} {s.min():>9.3f} {s.max():>9.3f}')

print('\n--- Corrélations entre métriques ---')
r_mb, p_mb = pearsonr(df_gpt['medcon_f1'], df_gpt['bleu'])
print(f'  MEDCON <-> BLEU  : r = {r_mb:+.3f}  (p={p_mb:.4f})')
if USE_COMET:
    r_mc, p_mc = pearsonr(df_gpt['medcon_f1'], df_gpt['comet'])
    r_bc, p_bc = pearsonr(df_gpt['bleu'],      df_gpt['comet'])
    print(f'  MEDCON <-> COMET : r = {r_mc:+.3f}  (p={p_mc:.4f})')
    print(f'  BLEU   <-> COMET : r = {r_bc:+.3f}  (p={p_bc:.4f})')

COMPARAISON MEDCON vs BLEU vs COMET  (GPT-4o)

Métrique         Moyenne       Std       Min       Max
----------------------------------------------------
MEDCON F1          0.398     0.475     0.000     1.000
BLEU              49.152    12.643    18.333    79.768
COMET              0.875     0.034     0.710     0.914

--- Corrélations entre métriques ---
  MEDCON <-> BLEU  : r = +0.071  (p=0.6258)
  MEDCON <-> COMET : r = -0.029  (p=0.8407)
  BLEU   <-> COMET : r = +0.408  (p=0.0036)


## 8. Analyse des erreurs de terminologie

In [16]:
print('=' * 80)
print('ANALYSE DES ERREURS DE TERMINOLOGIE')
print('=' * 80 + '\n')

all_missed = []
for ml in df_gpt['missed']:
    all_missed.extend(ml)
missed_counter = Counter(all_missed)

print('TOP 30 PAIRES MANQUÉES (terme EN présent en source, terme FR absent en traduction) :')
for i, (lbl, count) in enumerate(missed_counter.most_common(30), 1):
    print(f'  {i:2d}. [{count:2d}x] {lbl}')

all_extra = []
for el in df_gpt['extra']:
    all_extra.extend(el)
extra_counter = Counter(all_extra)

print('\nTOP 30 PAIRES EXTRAS (terme FR dans traduction, terme EN absent de la source) :')
for i, (lbl, count) in enumerate(extra_counter.most_common(30), 1):
    print(f'  {i:2d}. [{count:2d}x] {lbl}')

print(f'\n  Paires manquées uniques : {len(missed_counter)}')
print(f'  Paires extras uniques   : {len(extra_counter)}')

ANALYSE DES ERREURS DE TERMINOLOGIE

TOP 30 PAIRES MANQUÉES (terme EN présent en source, terme FR absent en traduction) :
   1. [ 1x] genetics -> génétique
   2. [ 1x] cardiorespiratory fitness -> capacité cardiorespiratoire
   3. [ 1x] cardiometabolic risk factors -> facteurs de risque cardiométabolique
   4. [ 1x] acetylcholinesterase -> acetylcholinesterase
   5. [ 1x] antimicrobial stewardship -> gestion responsable des antimicrobiens
   6. [ 1x] veterinary -> médecine vétérinaire
   7. [ 1x] deficiency -> carence nutritionnelle
   8. [ 1x] adverse effects -> effets indésirables
   9. [ 1x] anesthesiologists -> anesthésiologistes

TOP 30 PAIRES EXTRAS (terme FR dans traduction, terme EN absent de la source) :
   1. [ 3x] aftercare -> post-cure
   2. [ 2x] sadness -> tristesse
   3. [ 1x] diagnostic imaging -> imagerie diagnostique
   4. [ 1x] maternal health -> santé maternelle
   5. [ 1x] acquired immunodeficiency syndrome -> sida
   6. [ 1x] anesthetists -> anesthésistes

  Paire

## 9. Exemples : meilleur / médian / pire document

In [17]:
def afficher_doc(row, titre):
    print(f"\n{'#'*80}\n  {titre}  (doc #{row['doc_id']})\n{'#'*80}")
    print(f"\nSOURCE EN :\n{row['source_en'][:400]}...")
    print(f"\nTRADUCTION GPT-4o FR :\n{row['gpt_fr'][:400]}...")
    print(f"\n{'─'*60}")
    print(f"MEDCON  F1={row['medcon_f1']:.3f}  P={row['medcon_precision']:.3f}  R={row['medcon_recall']:.3f}")
    bleu_val = row.get('bleu', float('nan'))
    if not np.isnan(bleu_val):
        print(f"BLEU    {bleu_val:.1f}")
    print(f"Paires  attendues={row['n_expected']}  trouvées={row['n_found']}  matchées={row['n_match']}")
    print(f"\n[OK] MATCHÉES ({len(row['matched'])}) :")
    for c in row['matched'][:8]:  print(f'   {c}')
    print(f"\n[X]  MANQUÉES ({len(row['missed'])}) :")
    for c in row['missed'][:8]:   print(f'   {c}')
    print(f"\n[+]  EXTRAS   ({len(row['extra'])}) :")
    for c in row['extra'][:8]:    print(f'   {c}')


best_id   = df_gpt['medcon_f1'].idxmax()
worst_id  = df_gpt['medcon_f1'].idxmin()
median_id = (df_gpt['medcon_f1'] - df_gpt['medcon_f1'].median()).abs().idxmin()

afficher_doc(df_gpt.iloc[best_id],   'MEILLEUR DOCUMENT (F1 max)')
afficher_doc(df_gpt.iloc[median_id], 'DOCUMENT MÉDIAN')
afficher_doc(df_gpt.iloc[worst_id],  'PIRE DOCUMENT (F1 min)')


################################################################################
  MEILLEUR DOCUMENT (F1 max)  (doc #6)
################################################################################

SOURCE EN :
Ticks can carry multiple pathogens, and Inner Mongolia's animal husbandry provides excellent environmental conditions for ticks. This study characterized the microbiome of ticks from different geographical locations in Inner Mongolia; 905 Dermacentor nuttalli and 36 Ixodes persulcatus were collected from sheep in three main pasture areas and from bushes within the forested area. Mixed DNA samples ...

TRADUCTION GPT-4o FR :
Les tiques peuvent transporter plusieurs agents pathogènes, et l'élevage animal en Mongolie intérieure offre d'excellentes conditions environnementales pour les tiques. Cette étude a caractérisé le microbiome des tiques provenant de différents lieux géographiques en Mongolie intérieure ; 905 Dermacentor nuttalli et 36 Ixodes persulcatus ont été collectés 